<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/05_BigQuery_Graph_and_BQCA_Agent.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F05_BigQuery_Graph_and_BQCA_Agent.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/05_BigQuery_Graph_and_BQCA_Agent.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/05_BigQuery_Graph_and_BQCA_Agent.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/05_BigQuery_Graph_and_BQCA_Agent.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

---

# Track 1 (Notebook 05 — Lab 1.5): BigQuery Property Graph (ISO GQL) & BigQuery Conversational Analytics (BQCA) Data Agent
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

<p align="center">
  <img src="https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook5_bq_graph_and_bqca_flow.png" alt="Track 1 Notebook 05 Architecture: BigQuery Property Graph & Conversational Analytics Agent" width="95%">
</p>

---

### 📋 Notebook 05 Step-by-Step Execution Summary
1. **Step 0 (Environment Setup & APIs)**: Auto-detect active `PROJECT_ID` and Singapore region (`asia-southeast1`), and enable `bigquery.googleapis.com`, `geminidataanalytics.googleapis.com`, `cloudaicompanion.googleapis.com`, and `dataplex.googleapis.com`.
2. **Step 1 (Invoke `.sql` Bootstrap Files)**: Ensure `acsm_bronze`, `acsm_silver`, and `acsm_gold` exist (so this notebook can run standalone) and execute [`06_bigquery_graph_and_bqca.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql) to materialize the Gold Semantic Marts (`gold_underwriting_funnel`, `gold_collections_risk`), the 4 Graph Node Tables + 3 Graph Edge Tables, and `CREATE OR REPLACE PROPERTY GRAPH acsm_gold.acsm_credit_ecosystem_graph`.
3. **Step 2 (Part A.1 — Inspect BigQuery Property Graph Schema & Cardinality)**: Audit the nodes (`Customer`, `CreditFacility`, `Merchant`, `EmployerSegment`) and edges (`HOLDS_FACILITY`, `TRANSACTED_AT`, `WORKS_IN_SEGMENT`) of `acsm_gold.acsm_credit_ecosystem_graph` and explore the interactive **BigQuery Graph Visualizer** in the Cloud Console.
4. **Step 3 (Part A.2 — ISO GQL Query 1: 2-Hop Customer $\rightarrow$ CreditFacility & Merchant Portfolio Traversal)**: Query `GRAPH_TABLE(acsm_gold.acsm_credit_ecosystem_graph MATCH ...)` to profile multi-product customers across Malaysian states.
5. **Step 4 (Part A.3 — ISO GQL Query 2: Multi-Hop First-Party Fraud & Delinquency Contagion Ring Detection)**: Traverse `(c1:Customer)-[:TRANSACTED_AT]->(m:Merchant)<-[:TRANSACTED_AT]-(c2:Customer)` where both customers have active delinquency (`is_delinquent = TRUE`) to detect merchant-linked delinquency clusters without complex relational self-joins.
6. **Step 5 (Part A.4 — ISO GQL Query 3: AEON Ecosystem Cross-Sell Path Discovery)**: Traverse from pre-qualified Easy Payment (`EP`) customers with zero delinquency, low DSR, and active PDPA consent (`pdpa_marketing_consent = 'Y'`) who hold zero active Credit Cards (`active_card_count = 0`) through `TRANSACTED_AT` edges to identify top AEON Privilege Merchants for instant Credit Card cross-sell.
7. **Step 6 (Part B.1 — Create & Publish BigQuery Conversational Analytics Data Agents)**: Programmatically provision and publish governed BQCA Data Agents (`acsm-aeon360-bqca-agent` & `acsm-credit-graph-bqca-agent`) via `geminidataanalytics.googleapis.com/v1beta`, grounded on `acsm_gold` tables, Dataplex Knowledge Graph `schemaRelationships`, ACSM Domain Glossary (`CTOS`, `AKPK`, `DSR`, `Unpaid_OSP`, `FinPlus`), Verified Golden SQL/GQL queries, and `BigQueryPropertyGraphReference` (`C1.1.2.9`, `C1.1.2.10`, `M1.4.2`, `M1.4.4`).
8. **Step 7 & Step 8 (Part B.2 & B.3 — Multi-Turn Conversational Q&A over Gold Marts & Property Graph)**: Execute live natural-language business and graph-traversal questions against the BQCA Agent (`:chat` API) returning governed executive answers, transparent SQL/GQL queries, and live BigQuery result frames.

---
## Step 0: Configure Parameters (`PROJECT_ID` & Singapore Region) and Enable BigQuery & Gemini Data Analytics APIs
This notebook dynamically detects your active Google Cloud project (`PROJECT_ID`) and runs in **`asia-southeast1` (Singapore)** (while using the `global` control plane endpoint for the **Gemini Data Analytics / BigQuery Conversational Analytics API** `geminidataanalytics.googleapis.com`).

In [ ]:
# @title Step 0: Auto-Detect `PROJECT_ID`, Enable BigQuery & Gemini Data Analytics APIs
import os
import subprocess
import google.auth

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}
GDA_LOCATION = "global"  # @param {type:"string"}
REPO_URL = "https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git"
REPO_DIR = "aeon-credit-gcp-workshop"

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    _, PROJECT_ID = google.auth.default()

BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["GDA_LOCATION"] = GDA_LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

%load_ext google.cloud.bigquery

!gcloud config set project $PROJECT_ID
!gcloud services enable \
    bigquery.googleapis.com \
    geminidataanalytics.googleapis.com \
    cloudaicompanion.googleapis.com \
    dataplex.googleapis.com \
    storage.googleapis.com \
    --project=$PROJECT_ID

![ -d aeon-credit-gcp-workshop ] || git clone --depth 1 $REPO_URL $REPO_DIR
!git -C $REPO_DIR pull --ff-only
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

print(f"✅ Active Project             : {PROJECT_ID}")
print(f"✅ BigQuery Region            : {LOCATION} (Singapore)")
print(f"✅ Gemini Data Analytics Host : geminidataanalytics.googleapis.com ({GDA_LOCATION})")

---
## Step 1: Invoke `.sql` Bootstrap Files (`acsm_bronze`, `acsm_silver`, `acsm_gold` & BigQuery Property Graph)
All SQL DDL transformations for the Gold Semantic Marts (`gold_underwriting_funnel`, `gold_collections_risk`), the Property Graph Node & Edge tables, and the `CREATE OR REPLACE PROPERTY GRAPH acsm_gold.acsm_credit_ecosystem_graph` statement are stored in [`track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql).

> **Standalone Execution Support**: If `acsm_bronze` or `acsm_silver` have not been populated yet in this project, Step 1.1 automatically executes `00_create_8_tables_ddl_with_descriptions.sql`, `01_load_data_from_gcs.sql`, and `02_medallion_and_reconciliation.sql` first.

In [ ]:
# @title Step 1.1: Invoke `.sql` Files to Build `acsm_gold` Semantic Marts & `acsm_credit_ecosystem_graph`
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_bronze.Fact_CC_Sales >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql && \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_data_from_gcs.sql \
)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_gold.gold_aeon_customer360_profile >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/02_medallion_and_reconciliation.sql \
)
!bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/06_bigquery_graph_and_bqca.sql
print("✅ Executed 06_bigquery_graph_and_bqca.sql (`acsm_gold.acsm_credit_ecosystem_graph` & Gold BQCA Marts ready)!")

---
## Step 2 (Part A.1): Inspect the BigQuery Property Graph Schema & Node/Edge Cardinality
**BigQuery Property Graph (`CREATE OR REPLACE PROPERTY GRAPH`)** allows ACSM to model complex many-to-many financial relationships directly on top of governed BigQuery tables in `acsm_gold`—with **zero ETL export to external graph databases** and full inheritance of BigQuery's IAM, Dataplex lineage, and BNM RMiT regional governance (`asia-southeast1`).

### Graph Topology: `acsm_gold.acsm_credit_ecosystem_graph`
| Graph Element Type | Label | Underlying `acsm_gold` Table | Key Identifier(s) | Business Role in ACSM Ecosystem |
|---|---|---|---|---|
| **Node** | `Customer` | `graph_node_customer` | `KEY (cif_id)` | Deduplicated `m3CIF` customer profile with state, income, `wallet_tier`, `akpk_status`, `combined_unpaid_osp_myr`, and `is_delinquent` flag |
| **Node** | `CreditFacility` | `graph_node_credit_facility` | `KEY (facility_id)` | Easy Payment (`EP_*`) and Credit Card (`CC_*`) facilities with `credit_score`, `score_rank`, `new_dsr`, and `exposure_amount_myr` |
| **Node** | `Merchant` | `graph_node_merchant` | `KEY (merchant_id)` | Retail spend locations (`LDESC`) and `PriviledgeMerchantsGrp` across `Fact_CC_Sales` and `Fact_EP_Sales` |
| **Node** | `EmployerSegment` | `graph_node_employer_segment` | `KEY (employer_segment_id)` | Employer / Nature-of-Business (`NOB`) + State clusters (`NOB\|State`) |
| **Edge** | `HOLDS_FACILITY` | `graph_edge_holds_facility` | `(Customer)-[HOLDS_FACILITY]->(CreditFacility)` | Links customers to their approved/applied EP and CC credit facilities |
| **Edge** | `TRANSACTED_AT` | `graph_edge_transacted_at` | `(Customer)-[TRANSACTED_AT]->(Merchant)` | Links customers to retail merchants with aggregated `total_spend_myr` and `tx_count` |
| **Edge** | `WORKS_IN_SEGMENT` | `graph_edge_works_in_segment` | `(Customer)-[WORKS_IN_SEGMENT]->(EmployerSegment)` | Links customers to shared employment/industry and state segments |

> 🖥️ **Viewing in BigQuery Studio Graph Visualizer**:
> 1. Open **BigQuery Studio** in the Google Cloud Console (`asia-southeast1`).
> 2. Expand **`acsm_gold` $\rightarrow$ Property Graphs $\rightarrow$ `acsm_credit_ecosystem_graph`**.
> 3. Click the **Schema** tab to inspect the visual Node-and-Edge topology, or run any `GRAPH_TABLE` query in the SQL Editor and switch the results pane from **Table** to **Graph** view.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Audit Node and Edge Cardinality of `acsm_gold.acsm_credit_ecosystem_graph`
SELECT '1. NODE: Customer' AS graph_element, 'acsm_gold.graph_node_customer' AS backing_table, COUNT(*) AS row_count FROM `acsm_gold.graph_node_customer`
UNION ALL
SELECT '2. NODE: CreditFacility', 'acsm_gold.graph_node_credit_facility', COUNT(*) FROM `acsm_gold.graph_node_credit_facility`
UNION ALL
SELECT '3. NODE: Merchant', 'acsm_gold.graph_node_merchant', COUNT(*) FROM `acsm_gold.graph_node_merchant`
UNION ALL
SELECT '4. NODE: EmployerSegment', 'acsm_gold.graph_node_employer_segment', COUNT(*) FROM `acsm_gold.graph_node_employer_segment`
UNION ALL
SELECT '5. EDGE: HOLDS_FACILITY (Customer -> CreditFacility)', 'acsm_gold.graph_edge_holds_facility', COUNT(*) FROM `acsm_gold.graph_edge_holds_facility`
UNION ALL
SELECT '6. EDGE: TRANSACTED_AT (Customer -> Merchant)', 'acsm_gold.graph_edge_transacted_at', COUNT(*) FROM `acsm_gold.graph_edge_transacted_at`
UNION ALL
SELECT '7. EDGE: WORKS_IN_SEGMENT (Customer -> EmployerSegment)', 'acsm_gold.graph_edge_works_in_segment', COUNT(*) FROM `acsm_gold.graph_edge_works_in_segment`
ORDER BY graph_element;

---
## Step 3 (Part A.2 — ISO GQL Query 1): 2-Hop Customer $\rightarrow$ CreditFacility & Merchant Portfolio Traversal
Using standard **ISO GQL (`GRAPH_TABLE` & `MATCH`)** embedded inside BigQuery SQL, we traverse 2 hops simultaneously from a `Customer` node to both their `CreditFacility` (`EP` or `CC`) and the `Merchant` locations where they spend:
```
(f:CreditFacility) <-[h:HOLDS_FACILITY]- (c:Customer) -[t:TRANSACTED_AT]-> (m:Merchant)
```
This query identifies which **Privilege Merchant Groups** and **Malaysian States** generate the highest retail spend from ACSM credit facility holders, alongside their average Debt Service Ratio (`new_dsr`) and unpaid delinquency rate.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- ISO GQL Query 1: 2-Hop Traversal across (CreditFacility)<-[HOLDS_FACILITY]-(Customer)-[TRANSACTED_AT]->(Merchant)
SELECT
  state,
  product_line,
  merchant_group,
  COUNT(DISTINCT cif_id) AS active_customers,
  COUNT(DISTINCT facility_id) AS credit_facilities_held,
  ROUND(AVG(new_dsr), 2) AS avg_facility_dsr_pct,
  ROUND(SUM(exposure_amount_myr), 2) AS total_credit_exposure_myr,
  ROUND(SUM(total_spend_myr), 2) AS total_merchant_spend_myr,
  COUNTIF(is_delinquent) AS delinquent_customers_in_path
FROM GRAPH_TABLE(
  `acsm_gold.acsm_credit_ecosystem_graph`
  MATCH (f:CreditFacility)<-[h:HOLDS_FACILITY]-(c:Customer)-[t:TRANSACTED_AT]->(m:Merchant)
  COLUMNS (
    c.cif_id AS cif_id,
    c.state AS state,
    c.is_delinquent AS is_delinquent,
    f.facility_id AS facility_id,
    f.product_line AS product_line,
    f.exposure_amount_myr AS exposure_amount_myr,
    f.new_dsr AS new_dsr,
    t.total_spend_myr AS total_spend_myr,
    m.merchant_group AS merchant_group
  )
)
GROUP BY state, product_line, merchant_group
ORDER BY total_merchant_spend_myr DESC
LIMIT 15;

---
## Step 4 (Part A.3 — ISO GQL Query 2): Multi-Hop First-Party Fraud & Delinquency Contagion Ring Detection
In consumer credit and installment financing, **first-party synthetic fraud and merchant-collusion rings** often manifest as multiple delinquent borrowers (`is_delinquent = TRUE`) from the same **Employer/Industry Segment (`EmployerSegment`)** funneling high-value purchases through the same **Merchant (`Merchant`)**:
```
(c1:Customer)-[:TRANSACTED_AT]->(m:Merchant)<-[:TRANSACTED_AT]-(c2:Customer)
(c1)-[:WORKS_IN_SEGMENT]->(e:EmployerSegment)<-[:WORKS_IN_SEGMENT]-(c2)
```
Writing this as a multi-way self-join in relational SQL is error-prone, whereas **ISO GQL `MATCH`** expresses the cyclical diamond pattern declaratively in 3 lines.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- ISO GQL Query 2: Detect Delinquency Contagion & Collusion Rings Sharing Both Merchant AND Employer Segment
SELECT
  merchant_name,
  merchant_group,
  employer_segment_id,
  COUNT(*) AS connected_delinquent_pairs,
  COUNT(DISTINCT cif_1) + COUNT(DISTINCT cif_2) AS distinct_ring_members,
  ROUND(AVG(pair_unpaid_osp_myr), 2) AS avg_pair_unpaid_osp_myr,
  ROUND(SUM(pair_merchant_spend_myr), 2) AS total_ring_merchant_spend_myr
FROM GRAPH_TABLE(
  `acsm_gold.acsm_credit_ecosystem_graph`
  MATCH (c1:Customer)-[t1:TRANSACTED_AT]->(m:Merchant)<-[t2:TRANSACTED_AT]-(c2:Customer),
        (c1)-[w1:WORKS_IN_SEGMENT]->(e:EmployerSegment)<-[w2:WORKS_IN_SEGMENT]-(c2)
  WHERE c1.cif_id < c2.cif_id
    AND c1.is_delinquent = TRUE
    AND c2.is_delinquent = TRUE
  COLUMNS (
    c1.cif_id AS cif_1,
    c2.cif_id AS cif_2,
    m.merchant_name AS merchant_name,
    m.merchant_group AS merchant_group,
    e.employer_segment_id AS employer_segment_id,
    (c1.combined_unpaid_osp_myr + c2.combined_unpaid_osp_myr) AS pair_unpaid_osp_myr,
    (t1.total_spend_myr + t2.total_spend_myr) AS pair_merchant_spend_myr
  )
)
GROUP BY merchant_name, merchant_group, employer_segment_id
ORDER BY connected_delinquent_pairs DESC, avg_pair_unpaid_osp_myr DESC
LIMIT 15;

---
## Step 5 (Part A.4 — ISO GQL Query 3): AEON Ecosystem Cross-Sell Path Discovery (`Easy Payment` $\rightarrow$ `Credit Card`)
ACSM's strategic **AEON 360** growth objective is converting reliable **Easy Payment (`EP`)** borrowers who actively shop at **AEON Privilege Merchants** into **AEON Credit Card (`CC`)** holders—while strictly respecting **Malaysian PDPA marketing consent (`pdpa_marketing_consent = 'Y'`)** and BNM responsible lending affordability (`new_dsr <= 50`, `is_delinquent = FALSE`).

Below, we traverse:
```
(f:CreditFacility {product_line: 'EP'}) <-[:HOLDS_FACILITY]- (c:Customer {active_card_count: 0, is_delinquent: FALSE}) -[:TRANSACTED_AT]-> (m:Merchant)
```
to rank the top **Merchant Locations** and **Customer Candidates** for instant Credit Card cross-sell at point-of-sale.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- ISO GQL Query 3: Discover Pre-Qualified EP-to-CC Cross-Sell Candidates via Graph Traversal
SELECT
  cif_id,
  customer_name,
  state,
  occupation,
  net_income_myr,
  ep_facility_id,
  ep_exposure_myr,
  ep_new_dsr,
  merchant_name,
  merchant_group,
  ep_merchant_spend_myr
FROM GRAPH_TABLE(
  `acsm_gold.acsm_credit_ecosystem_graph`
  MATCH (f:CreditFacility)<-[h:HOLDS_FACILITY]-(c:Customer)-[t:TRANSACTED_AT]->(m:Merchant)
  WHERE f.product_line = 'EP'
    AND c.active_card_count = 0
    AND c.is_delinquent = FALSE
    AND c.pdpa_marketing_consent = 'Y'
    AND c.net_income_myr >= 4000
    AND f.new_dsr <= 50
  COLUMNS (
    c.cif_id AS cif_id,
    c.customer_name AS customer_name,
    c.state AS state,
    c.occupation AS occupation,
    c.net_income_myr AS net_income_myr,
    f.facility_id AS ep_facility_id,
    f.exposure_amount_myr AS ep_exposure_myr,
    f.new_dsr AS ep_new_dsr,
    m.merchant_name AS merchant_name,
    m.merchant_group AS merchant_group,
    t.total_spend_myr AS ep_merchant_spend_myr
  )
)
ORDER BY ep_merchant_spend_myr DESC, net_income_myr DESC
LIMIT 20;

---
## Step 6 (Part B.1): Create & Publish BigQuery Conversational Analytics (BQCA) Data Agents (`C1.1.2.9`, `C1.1.2.10`, `M1.4.2`, `M1.4.4`)
Today, **65% of ACSM's data users are non-technical business analysts** across Credit Risk, Collections, Marketing, and Finance who submit ~150 ad-hoc SQL request tickets per day.

Using the **BigQuery Conversational Analytics (BQCA) / Gemini Data Analytics API (`geminidataanalytics.googleapis.com/v1beta`)**, we programmatically provision and publish two governed Data Agents directly from [`track3_self_serve_analytics/bqca_agent_config.yaml`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track3_self_serve_analytics/bqca_agent_config.yaml):

1. **`acsm-aeon360-bqca-agent` (Relational Gold Marts + Knowledge Graph + Glossary + Verified Golden SQL)**:
   - **Data Sources (`tableReferences`)**: `acsm_gold.gold_aeon_customer360_profile`, `acsm_gold.gold_underwriting_funnel`, `acsm_gold.gold_collections_risk`, and `acsm_silver.recon_audit_log`.
   - **Dataset Knowledge Graph (`schemaRelationships`)**: Explicitly wires the `CIF_ID` join paths discovered by the Dataplex Data Governance & Insights Agent in **Notebook 03**.
   - **ACSM Domain Glossary (`glossaryTerms`)**: Defines Malaysian credit terminology (`EP`, `CC`, `CIF_ID`, `Unpaid_OSP`, `AKPK`, `DSR`, `FinPlus`, `PDPA Consent`).
   - **Verified Golden SQL (`exampleQueries`)**: Ensures 100% deterministic financial formulas (`Collection Efficiency Ratio = SAFE_DIVIDE(SUM(collection_osp_myr), NULLIF(SUM(billing_osp_myr), 0))`).
2. **`acsm-credit-graph-bqca-agent` (BigQuery Property Graph Grounded Agent)**:
   - **Data Source (`propertyGraphReferences`)**: Connects directly to `acsm_gold.acsm_credit_ecosystem_graph` (`BigQueryPropertyGraphReference`) with ISO GQL (`GRAPH_TABLE`) golden queries so analysts can ask multi-hop relationship questions in plain English!

> 🖥️ **How to Access Your Published BQCA Agents in BigQuery Studio UI**:
> 1. In the Google Cloud Console, navigate to **BigQuery $\rightarrow$ Studio** or **Agents** in the left navigation bar.
> 2. Click **Conversational Analytics** (or **+ New $\rightarrow$ Conversation**).
> 3. Select **`ACSM AEON360 Conversational Analytics & Graph Agent`** (`acsm-aeon360-bqca-agent`) or **`ACSM Credit Ecosystem Property Graph Agent`** (`acsm-credit-graph-bqca-agent`) and start asking questions in English or Bahasa Melayu!

In [ ]:
# @title Step 6.1: Provision & Publish `acsm-aeon360-bqca-agent` and `acsm-credit-graph-bqca-agent` via `geminidataanalytics.googleapis.com/v1beta`
import json
import time
import yaml
import google.auth
from google.auth.transport.requests import AuthorizedSession

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_session = AuthorizedSession(credentials)

with open("aeon-credit-gcp-workshop/track3_self_serve_analytics/bqca_agent_config.yaml", "r") as f:
    bqca_cfg = yaml.safe_load(f)

GDA_BASE_URL = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/{GDA_LOCATION}"

# 1. Build Context for Relational Gold Marts BQCA Agent (`acsm-aeon360-bqca-agent`)
relational_table_refs = []
for fq_table in bqca_cfg["target_tables"]:
    ds_id, tbl_id = fq_table.split(".")
    relational_table_refs.append({
        "projectId": PROJECT_ID,
        "datasetId": ds_id,
        "tableId": tbl_id,
    })

glossary_terms = [
    {"displayName": "EP", "description": "Easy Payment installment financing for motorcycles, appliances, and personal financing.", "labels": ["product", "easy_payment"]},
    {"displayName": "CC", "description": "AEON Credit Card products across Platinum, Gold, Silver, and Basic tiers.", "labels": ["product", "credit_card"]},
    {"displayName": "Unpaid_OSP", "description": "Overdue unpaid principal balance in Malaysian Ringgit (MYR).", "labels": ["collections", "delinquency"]},
    {"displayName": "Collection Efficiency Ratio", "description": "Ratio of collected principal to billed principal: SAFE_DIVIDE(SUM(collection_osp_myr), NULLIF(SUM(billing_osp_myr), 0)).", "labels": ["collections", "kpi"]},
    {"displayName": "AKPK", "description": "Agensi Kaunseling dan Pengurusan Kredit (Malaysian debt restructuring program; akpk_indicator = 'Y').", "labels": ["collections", "bnm"]},
    {"displayName": "DSR", "description": "Debt Service Ratio measuring monthly debt obligations divided by net monthly income.", "labels": ["underwriting", "affordability"]},
    {"displayName": "FinPlus", "description": "ACSM internal e-credit evaluation tier (finplus_tier).", "labels": ["underwriting", "collections"]},
]

schema_relationships = [
    {
        "leftSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_aeon_customer360_profile",
            "paths": ["CIF_ID"],
        },
        "rightSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_underwriting_funnel",
            "paths": ["cif_id"],
        },
        "sources": ["LLM_SUGGESTED"],
        "confidenceScore": 1.0,
    },
    {
        "leftSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_aeon_customer360_profile",
            "paths": ["CIF_ID"],
        },
        "rightSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_collections_risk",
            "paths": ["cif_id"],
        },
        "sources": ["LLM_SUGGESTED"],
        "confidenceScore": 1.0,
    },
]

example_queries = []
for ex in bqca_cfg["golden_sql_examples"]:
    # Qualify dataset references with PROJECT_ID so BQCA executes against the active project
    sql_text = ex["sql"].replace("`acsm_gold.", f"`{PROJECT_ID}.acsm_gold.").replace("`acsm_silver.", f"`{PROJECT_ID}.acsm_silver.")
    example_queries.append({
        "naturalLanguageQuestion": ex["question"],
        "sqlQuery": sql_text.strip(),
    })

relational_context = {
    "systemInstruction": bqca_cfg["system_instructions"],
    "datasourceReferences": {
        "bq": {
            "tableReferences": relational_table_refs,
        }
    },
    "glossaryTerms": glossary_terms,
    "schemaRelationships": schema_relationships,
    "exampleQueries": example_queries[:3],
    "options": {
        "chart": {"image": {"svg": {}}},
    },
}

def upsert_data_agent(agent_id: str, display_name: str, description: str, context_payload: dict) -> dict:
    agent_url = f"{GDA_BASE_URL}/dataAgents/{agent_id}"
    body = {
        "displayName": display_name,
        "description": description,
        "dataAnalyticsAgent": {
            "stagingContext": context_payload,
            "publishedContext": context_payload,
        },
    }
    check_resp = authed_session.get(agent_url)
    if check_resp.status_code == 200:
        resp = authed_session.patch(
            f"{agent_url}?updateMask=displayName,description,dataAnalyticsAgent.stagingContext,dataAnalyticsAgent.publishedContext",
            json=body,
        )
        action = "Updated"
    else:
        resp = authed_session.post(
            f"{GDA_BASE_URL}/dataAgents?dataAgentId={agent_id}",
            json=body,
        )
        action = "Created"

    data = resp.json()
    if resp.status_code in (200, 201):
        # Poll LRO if returned
        if "name" in data and "/operations/" in data["name"]:
            op_url = f"https://geminidataanalytics.googleapis.com/v1beta/{data['name']}"
            for _ in range(15):
                op_r = authed_session.get(op_url).json()
                if op_r.get("done"):
                    break
                time.sleep(2)
        print(f"✅ {action} BQCA Data Agent: projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}")
        return {"ok": True, "agent_name": f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}"}
    else:
        print(f"⚠️ {agent_id} response ({resp.status_code}): {json.dumps(data)[:300]}")
        return {"ok": False, "error": data}

# 1. Upsert Relational Gold Marts BQCA Agent
rel_result = upsert_data_agent(
    agent_id="acsm-aeon360-bqca-agent",
    display_name="ACSM AEON360 Conversational Analytics Agent",
    description="Governed ACSM Customer 360, Underwriting Funnel, and Collections Risk Conversational Analytics Agent in Singapore (asia-southeast1).",
    context_payload=relational_context,
)

# 2. Upsert BigQuery Property Graph BQCA Agent (`propertyGraphReferences` with fallback to Graph Node/Edge tables)
graph_context = {
    "systemInstruction": (
        bqca_cfg["system_instructions"]
        + "\nWhen answering relationship, merchant contagion, fraud ring, or multi-hop cross-sell questions, "
        + f"use ISO GQL GRAPH_TABLE(`{PROJECT_ID}.acsm_gold.acsm_credit_ecosystem_graph` MATCH ...) or the underlying graph_node_* and graph_edge_* tables in `{PROJECT_ID}.acsm_gold`."
    ),
    "datasourceReferences": {
        "bq": {
            "propertyGraphReferences": [
                {
                    "projectId": PROJECT_ID,
                    "datasetId": "acsm_gold",
                    "propertyGraphId": "acsm_credit_ecosystem_graph",
                }
            ]
        }
    },
    "glossaryTerms": glossary_terms,
    "exampleQueries": [example_queries[3]],
}

graph_result = upsert_data_agent(
    agent_id="acsm-credit-graph-bqca-agent",
    display_name="ACSM Credit Ecosystem Property Graph Agent",
    description="BigQuery Property Graph Conversational Agent grounded on acsm_gold.acsm_credit_ecosystem_graph for fraud ring detection and cross-sell path traversal.",
    context_payload=graph_context,
)

# Fallback if propertyGraphReferences preview is not allowlisted on the current project
if not graph_result["ok"]:
    print("ℹ️ Falling back to grounding `acsm-credit-graph-bqca-agent` on the 7 Graph Node/Edge tables with GQL Golden Queries...")
    graph_fallback_tables = [
        {"projectId": PROJECT_ID, "datasetId": "acsm_gold", "tableId": t}
        for t in [
            "graph_node_customer",
            "graph_node_credit_facility",
            "graph_node_merchant",
            "graph_node_employer_segment",
            "graph_edge_holds_facility",
            "graph_edge_transacted_at",
            "graph_edge_works_in_segment",
        ]
    ]
    graph_context["datasourceReferences"] = {"bq": {"tableReferences": graph_fallback_tables}}
    graph_result = upsert_data_agent(
        agent_id="acsm-credit-graph-bqca-agent",
        display_name="ACSM Credit Ecosystem Property Graph Agent",
        description="BigQuery Property Graph Conversational Agent grounded on acsm_gold graph node/edge tables and GRAPH_TABLE GQL.",
        context_payload=graph_context,
    )

---
## Step 7 (Part B.2): Live Multi-Turn Conversational Q&A with the BQCA Data Agent (`:chat` API)
Below, we define a helper function `ask_acsm_bqca_agent(...)` that sends natural-language business questions to `https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/global:chat` using our published **`acsm-aeon360-bqca-agent`** (with inline context fallback if needed).

For every business question, the BQCA Agent automatically:
1. Grounds the natural-language prompt against the **ACSM Domain Glossary**, **Dataplex Knowledge Graph `schemaRelationships`**, and **Verified Golden SQL** examples.
2. Generates and executes governed **BigQuery SQL** in `asia-southeast1`.
3. Returns the **Generated SQL Query** (for 100% auditability under BNM RMiT) alongside the **Natural-Language Executive Summary** and **Result Table**.

In [ ]:
# @title Step 7.1: Execute Multi-Turn Natural-Language Business Q&A via BQCA `:chat` API
from IPython.display import display, Markdown
from google.cloud import bigquery
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

def ask_acsm_bqca_agent(question: str, agent_id: str = "acsm-aeon360-bqca-agent", fallback_context: dict = None, fallback_sql: str = None):
    display(Markdown(f"### 💬 User Question to `{agent_id}`\n> **\"{question}\"**"))
    chat_url = f"{GDA_BASE_URL}:chat"
    agent_resource = f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}"

    payload = {
        "parent": f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}",
        "messages": [
            {
                "userMessage": {
                    "text": question
                }
            }
        ],
        "dataAgentContext": {
            "dataAgent": agent_resource,
            "contextVersion": "PUBLISHED",
        },
    }

    resp = authed_session.post(chat_url, json=payload)
    if resp.status_code != 200 and fallback_context is not None:
        # Fallback to stateless inlineContext if agent publishing is still propagating
        payload.pop("dataAgentContext", None)
        payload["inlineContext"] = fallback_context
        resp = authed_session.post(chat_url, json=payload)

    generated_sql = None
    final_text_parts = []

    if resp.status_code == 200:
        try:
            events = resp.json()
            if isinstance(events, dict):
                events = [events]
            for ev in events:
                sys_msg = ev.get("systemMessage", {})
                # Extract generated SQL
                if "data" in sys_msg:
                    data_msg = sys_msg["data"]
                    if "generatedSql" in data_msg:
                        generated_sql = data_msg["generatedSql"]
                    elif "query" in data_msg and "sql" in data_msg["query"]:
                        generated_sql = data_msg["query"]["sql"]
                # Extract final narrative text
                if "text" in sys_msg:
                    parts = sys_msg["text"].get("parts", [])
                    final_text_parts.extend(parts)
        except Exception as e:
            print(f"ℹ️ Parsed streaming response ({e})")
    else:
        print(f"ℹ️ BQCA API returned status {resp.status_code}; executing verified Golden SQL directly on BigQuery.")

    sql_to_run = generated_sql or fallback_sql
    if sql_to_run:
        display(Markdown(f"#### 🔍 BQCA Generated & Verified SQL (`{LOCATION}`)\n```sql\n{sql_to_run.strip()}\n```"))
        try:
            df = bq_client.query(sql_to_run).to_dataframe()
            display(df.head(10))
        except Exception as qe:
            print(f"⚠️ Query execution note: {qe}")

    if final_text_parts:
        display(Markdown("#### 🤖 BQCA Agent Executive Summary\n" + "\n\n".join(final_text_parts)))
    print("-" * 100)

# Question 1: Collections Risk & Efficiency across Malaysian States (RFP Clause C1.1.2.9)
ask_acsm_bqca_agent(
    question="What is our total Billing OSP, Collection OSP, Unpaid OSP, and Collection Efficiency Ratio across Malaysian states?",
    agent_id="acsm-aeon360-bqca-agent",
    fallback_context=relational_context,
    fallback_sql=example_queries[0]["sqlQuery"],
)

# Question 2: Underwriting Funnel Comparison between Easy Payment (EP) and Credit Cards (CC)
ask_acsm_bqca_agent(
    question="Compare credit application volume, average credit score, and average DSR between Easy Payment (EP) and Credit Cards (CC) by channel and decision status.",
    agent_id="acsm-aeon360-bqca-agent",
    fallback_context=relational_context,
    fallback_sql=example_queries[1]["sqlQuery"],
)

# Question 3: PDPA-Compliant EP-to-CC Cross-Sell Target List
ask_acsm_bqca_agent(
    question="Find top Easy Payment (EP) customers with zero unpaid delinquency, net income above RM 5,000, and PDPA marketing consent who do not hold an active Credit Card yet.",
    agent_id="acsm-aeon360-bqca-agent",
    fallback_context=relational_context,
    fallback_sql=example_queries[2]["sqlQuery"],
)

---
## Step 8 (Part B.3): Natural-Language Graph Traversal Q&A with `acsm-credit-graph-bqca-agent`
Now we query our **Property Graph-Grounded BQCA Agent (`acsm-credit-graph-bqca-agent`)** in plain English to identify **Merchant Delinquency Contagion** across `acsm_gold.acsm_credit_ecosystem_graph`.

In [ ]:
# @title Step 8.1: Ask a Natural-Language Graph Traversal Question via `acsm-credit-graph-bqca-agent`
ask_acsm_bqca_agent(
    question="Which merchants have the highest number of delinquent customers transacting there in our BigQuery Property Graph, and what is their linked unpaid principal (OSP)?",
    agent_id="acsm-credit-graph-bqca-agent",
    fallback_context=graph_context,
    fallback_sql=example_queries[3]["sqlQuery"],
)

---
## ✅ Track 1 (Notebook 05 — Lab 1.5) Summary

| Lab 1.5 Section | Capability Demonstrated | Key BigQuery / Gemini Data Analytics Artefacts (`asia-southeast1`) |
|---|---|---|
| **Part A.1: BigQuery Property Graph DDL** | Zero-ETL Property Graph over governed relational tables (`CREATE OR REPLACE PROPERTY GRAPH`) | `acsm_gold.acsm_credit_ecosystem_graph` (`Customer`, `CreditFacility`, `Merchant`, `EmployerSegment` nodes + `HOLDS_FACILITY`, `TRANSACTED_AT`, `WORKS_IN_SEGMENT` edges) |
| **Part A.2: ISO GQL 2-Hop Portfolio Traversal** | Declarative `GRAPH_TABLE` & `MATCH` pattern across facilities, customers, and merchants | `(f:CreditFacility)<-[:HOLDS_FACILITY]-(c:Customer)-[:TRANSACTED_AT]->(m:Merchant)` |
| **Part A.3: Fraud & Delinquency Contagion Rings** | Cyclical diamond graph pattern detecting delinquent customer pairs sharing both a merchant and employer segment | `(c1:Customer)-[:TRANSACTED_AT]->(m:Merchant)<-[:TRANSACTED_AT]-(c2:Customer)` + `WORKS_IN_SEGMENT` |
| **Part A.4: AEON 360 Cross-Sell Path Discovery** | Multi-hop discovery of PDPA-consented, zero-delinquency Easy Payment customers transacting at privilege merchants without an active Credit Card | `GRAPH_TABLE` filtered by `active_card_count = 0`, `is_delinquent = FALSE`, `pdpa_marketing_consent = 'Y'`, `new_dsr <= 50` |
| **Part B.1: BQCA Data Agent Provisioning** | Programmatic creation & publishing of relational and graph-grounded Data Agents via `geminidataanalytics.googleapis.com/v1beta` | `acsm-aeon360-bqca-agent` (`tableReferences` + `schemaRelationships` + `glossaryTerms` + `exampleQueries`) & `acsm-credit-graph-bqca-agent` (`propertyGraphReferences`) |
| **Part B.2 & B.3: Live Multi-Turn Conversational Q&A** | Self-service Natural-Language to SQL/GQL & chart generation for ACSM's 65% non-technical business cohort (`C1.1.2.9`, `C1.1.2.10`, `M1.4.2`, `M1.4.4`) | Live `:chat` invocations returning auditable SQL/GQL, executive summaries, and BigQuery DataFrames |